# 15 — Text as a Sequence: Preparing Sentences for an RNN

**Learning objective:** See exactly how a natural-language sentence becomes a tensor sequence that a recurrent network can process.

This notebook extends the RNN track from numeric time series to **token sequences**:

```text
raw sentence
   ↓
tokens
   ↓
vocabulary IDs
   ↓
fixed-length padded sequence
   ↓
embedding vector at each timestep
   ↓
RNN hidden states
   ↓
sentence representation
```

The important idea is that RNNs do not read words directly. They receive **vectors over ordered timesteps**.

## 1. The text data contract

Our local dataset contains one sentence, one binary sentiment label, and the source domain.

- `label = 1`: positive sentiment
- `label = 0`: negative sentiment
- domains: Amazon, IMDb, Yelp

For an NLP RNN, **time** is token position rather than clock time.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

SEED=42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

DATA=Path("RNN/data/sentence_sentiment_uci.csv")
df=pd.read_csv(DATA)
print("TensorFlow:",tf.__version__)
print("shape:",df.shape)
print(df.groupby(["source","label"]).size().unstack())
display(df.sample(6,random_state=SEED))

## 2. Tokenization and integer IDs

`TextVectorization` learns a vocabulary. Each token becomes an integer index.

Changing the vocabulary size changes what is represented explicitly versus mapped to `[UNK]`.

Changing `output_sequence_length` changes how much sentence context is preserved before truncation and how much padding is added to shorter sentences.

In [ ]:
train_text=df.sample(frac=.70,random_state=SEED)["text"].astype(str).to_numpy()

vectorizer=tf.keras.layers.TextVectorization(
    max_tokens=5000,
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    output_mode="int",
    output_sequence_length=20,
)
vectorizer.adapt(train_text)

vocab=vectorizer.get_vocabulary()
print("vocabulary size:",len(vocab))
print("special tokens + first learned tokens:",vocab[:20])

examples=tf.constant([
    "The movie was excellent and beautifully acted.",
    "The battery was terrible and stopped working."
])
ids=vectorizer(examples)
print("integer tensor shape:",ids.shape)
print(ids.numpy())

In [ ]:
id_to_token=dict(enumerate(vocab))
for sentence,row in zip(examples.numpy(),ids.numpy()):
    decoded=[id_to_token.get(int(i),"?") for i in row if i!=0]
    print("\nraw:",sentence.decode())
    print("decoded vectorized sequence:",decoded)

## 3. Padding and masking

A batch needs rectangular tensors, but sentences have different lengths.

Padding uses ID `0`. With `Embedding(mask_zero=True)`, TensorFlow marks padded timesteps so compatible recurrent layers do not treat padding as meaningful language.

This is the text equivalent of saying: **these timesteps do not contain observations**.

In [ ]:
embed=tf.keras.layers.Embedding(
    input_dim=len(vocab),
    output_dim=8,
    mask_zero=True
)
embedded=embed(ids)
mask=embed.compute_mask(ids)

print("token IDs:",ids.shape)
print("embedded sequence:",embedded.shape)
print("mask:",mask.shape)
print(mask.numpy().astype(int))

## 4. Hidden state over tokens

For sentence classification we usually want one vector representing the whole sentence.

With `return_sequences=True`, a SimpleRNN exposes one hidden vector for every token position. The final valid hidden state can act as the sentence representation.

In [ ]:
rnn=tf.keras.layers.SimpleRNN(6,return_sequences=True,return_state=True)
all_states,final_state=rnn(embedded,mask=mask)

print("all hidden states:",all_states.shape)
print("final sentence state:",final_state.shape)
print("first sentence final representation:")
print(np.round(final_state.numpy()[0],3))

## Change map

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Increase sequence length | more tokens retained | more context, more recurrent steps, more compute |
| Reduce vocabulary | more tokens become `[UNK]` | information loss rises |
| Increase embedding dimension | richer token vectors | more parameters and capacity |
| Increase hidden units | larger sentence state | more memory/capacity and overfitting risk |
| Enable masking | padding is ignored | state represents real tokens rather than zeros |

## What you should now be able to explain

A sentence classifier is not magic:

`sentence → token IDs → embeddings → hidden-state updates → final sentence vector → classifier`.

The next notebook trains that complete pipeline in TensorFlow.